# Seguimiento de envíos locales

In [6]:
import pandas as pd
import os
import ipywidgets as widgets
from ipywidgets import  Layout,HTML
from IPython.display import display, Markdown, clear_output
import pickle
from datetime import datetime, timedelta, timezone, date
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Alignment, Font
from openpyxl.formatting.rule import CellIsRule
from tqdm.notebook import tqdm
from tkinter import Tk, filedialog as fd
import win32com.client
import warnings
import shutil

warnings.filterwarnings("ignore")
def open_file_selection(initialdir='',filter_name='*.*'):
    """
    Opens a file selection dialog and returns the selected file paths.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: A tuple of selected file paths or an empty tuple if no file is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top
    
    filetypes = (
        (filter_name,f"*{filter_name.split(' ')[0]}*.*"),
        ('All files', '*.*'),
        ('Excel', '*.xlsx'),
        ('CSV', '*.doc')
    )
    files = fd.askopenfilenames(
        filetypes=filetypes,
        initialdir=initialdir
    )
    
    # Destroy the root window after use
    root.destroy()
    
    return files
def select_directory(initialdir):
    """
    Opens a directory selection dialog and returns the selected directory path.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: The selected directory path as a string, or an empty string if no directory is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top

    # Open the directory selection dialog
    directory = fd.askdirectory(initialdir=initialdir)

    # Destroy the root window after use
    root.destroy()

    return directory

# Function to show a pop-up message
def show_popup_message(message):
    display(Markdown(f"### **{message}**"))
    

# Decorator to handle the permission error
def handle_permission_error_with_popup(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except PermissionError as e:
            if e.errno == 13:  # Permission denied error
                show_popup_message(f"Error: {e}\nFavor de cerrar el archivo.")
    return wrapper

def save_df(df, filepath, sheet_name, index=False):
    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index)

@handle_permission_error_with_popup
def save_df_multiple(df_dict=dict(), filepath='',index=False):
    with pd.ExcelWriter(filepath) as writer:
        for key in df_dict.keys():
            df_dict[key].to_excel(writer, sheet_name=key, index=index)

@handle_permission_error_with_popup
def save_wb(wb, filepath):
    wb.save(filepath)

@handle_permission_error_with_popup
def append_sheet(df,path,sheet_name,index):
    with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index) 


def is_file_open(filepath):
    # Check if the file exists
    if not os.path.exists(filepath):
        return False  # File does not exist, so treat it as "open" for your logic

    try:
        # Try to open the file in write mode
        with open(filepath, 'a'):
            pass
        return False  # File is not open (no exception raised)
    except PermissionError:
        return True 
    
def get_path(file_selectors,selector_name):
    file_selector=[file_selector for file_selector in file_selectors if file_selector.children[0].description[0:-1]==selector_name]
    if len(file_selector)==0:
        show_popup_message(f"No se encontro el selector")
        raise SystemExit()
    file_selector=file_selector[0]
    if not file_selector.children[1].value:
        return None
    selected_path=file_selector.children[1].value.strip()
    return selected_path

def read_excel(path=None,sheet_name=0,header=0):
    with pd.ExcelFile(path) as xls:
        df=pd.read_excel(path,sheet_name=sheet_name,header=header)
    return df

def verify_selections(file_selectors):
    not_selected=[]
    for file_selector in file_selectors:
        selector=file_selector.children[0].description[0:-1]
        selected=file_selector.children[1].value.strip()
        if not os.path.exists(selected):
            file_selector.children[1].value='Not selected'
            selected='Not selected'
        if selector not in mandatory_selectors:
            continue
        if selected=='Not selected':
            not_selected.append(selector)
            continue

    if len(not_selected)>0:
        show_popup_message(f'Favor de seleccionar los siguientes archivos:{not_selected}')
        raise SystemExit()

def close_xl_if_open(path):
    if is_file_open(path):
        try:
            excel = win32com.client.Dispatch("Excel.Application")
            workbook = excel.Workbooks(path)
            workbook.Save()
            workbook.Close()
        except:
            show_popup_message(f"Cerrar el archivo: {path}")
            raise SystemExit()               

##### Excel management functions
def find_cell_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                return cell.coordinate  # Return cell address (e.g., 'B2')
    return None 

def find_all_cells_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    cells=[]
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                cells.append(cell.coordinate)
    return cells 

def set_number_format(ws,col_name,format):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=find_cell_by_text(ws,col_name)
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].number_format = format

def fill_column(ws,col_name,fill):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].fill=fill

def font_column(ws,col_name,font):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].font=font    



def fill_formula(ws,col_name,formula):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=formula


# Cell colors
light_green='99FF99'
avocato_green='E2EFDA'
dark_blue='002060'
light_blue='DDEBF7'
grey='595959'
dark_red='9C0006'
melon='FCE4D6'
white='FFFFFF'
black='000000'
light_pink='FFC7CE'
light_yellow='FFF2CC'


# Save and load state using pickle
def save_state_pickle(state, filename='folder_state.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(state, f)

def load_state_pickle(filename='folder_state.pkl'):
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        return {"folder_input": None, "folder_output": None, "selections": {}}

def set_opposites(name_a,name_b):
    opposite = {name_a: name_b, name_b: name_a}
    return opposite
    

def update_oposite_selectors(filter_name,selector_name_a,selector_name_b):
    opposite = {selector_name_a: selector_name_b, selector_name_b: selector_name_a}
    if filter_name in [selector_name_a, selector_name_b]:
        # Remove the opposite action from mandatory selectors
        if opposite[filter_name] in mandatory_selectors:
            mandatory_selectors.remove(opposite[filter_name])
        # Add current filter name if it's not already in the list
        if filter_name not in mandatory_selectors:
            mandatory_selectors.append(filter_name)
        
        # Update file selectors to set the opposite field to 'Not selected'
        for file_selector in file_selectors:
            if file_selector.children[0].description[:-1] == opposite[filter_name]:
                file_selector.children[1].value = 'Not selected'
                state[opposite[filter_name]]=f" Not selected"
                save_state_pickle(state)
    
# Agregar las columnas criticas por archivo
filters = ['Edi master']


mandatory_cols={'Korrus':[
                'PurchaseOrder',
                'PODate',
                'REF02_ClearTextClause',
                'REF02_CustomerOrderNumber',
                'REF02_CarrierAccount',
                'TermsTypeCode',
                'TermsBasisDateCode',
                'description',
                'Routing',
                'JobName',
                'JobNameNotes',
                'ShipmentMarkings',
                'ShipmentMarkingsNotes',
                'ShipToParty',
                'ShipToAddress1',
                'ShipToAddress2',
                'ShipToCity',
                'ShipToState',
                'ShipToPostalCode',
                'ShipToCountryCode',
                'BillToParty',
                'Address1',
                'Address2',
                'BillToCity',
                'BillToState',
                'BillToPostalCode',
                'BillToCountryCode',
                'Supplier',
                'SupplierAddress1',
                'SupplierAddress2',
                'SupplierCity',
                'SupplierState',
                'SupplierPostalCode',
                'SupplierCountryCode',
                'LineNumber',
                'Quantity',
                'Uom',
                'price',
                'ProductServiceID',
                'StorageLocation',
                'AssignedDropZone'
                ],
                'EDI master':[
                'PO',
                'PODate',
                'REF02_ClearTextClause',
                'REF02_CustomerOrderNumber',
                'REF02_CarrierAccount',
                'TermsTypeCode',
                'TermsBasisDateCode',
                'description',
                'Routing',
                'JobName',
                'JobNameNotes',
                'ShipmentMarkings',
                'ShipmentMarkingsNotes',
                'ShipToParty',
                'ShipToAddress1',
                'ShipToAddress2',
                'ShipToCity',
                'ShipToState',
                'ShipToPostalCode',
                'ShipToCountryCode',
                'BillToParty',
                'Address1',
                'Address2',
                'BillToCity',
                'BillToState',
                'BillToPostalCode',
                'BillToCountryCode',
                'Supplier',
                'SupplierAddress1',
                'SupplierAddress2',
                'SupplierCity',
                'SupplierState',
                'SupplierPostalCode',
                'SupplierCountryCode',
                'LineNumber',
                'Quantity',
                'Uom',
                'price',
                'ProductServiceID',
                'StorageLocation',
                'AssignedDropZone'
                ]
                }
mandatory_selectors=[
    'Edi master',
    'Accesory List']

excluding_selectors=['Work Order Action','Independent Demands']

def check_mandatory_cols(cols,selector_name, raise_error=True):
    missing_columns = [col for col in mandatory_cols[selector_name] if col not in cols]
    if len(missing_columns)>0:
        show_popup_message(f"No se encontraron las siguientes columnas en el archivo {selector_name}: {missing_columns}")
        if raise_error:
            raise SystemExit()
        return False
    return True
state = load_state_pickle()
if ('folder_output' not in state.keys()):
    state['folder_output']=''

folder_output_button = widgets.Button(description="Folder de salidas:")
if state['folder_output']:
    initialdir=state['folder_output']
else:
    initialdir='Not selected'
folder_output_label = widgets.Label(value=initialdir)

def on_output_button_click(b):
    if state['folder_output']:
        initialdir=state['folder_output']
    else:
        initialdir='/'
    selected_dir = select_directory(initialdir=initialdir)
    if selected_dir:
        folder_output_label.value = f"{selected_dir}"
        state['folder_output']=selected_dir
        save_state_pickle(state)
    else:
        folder_output_label.value = "Not selected"

# Attach the event to the folder_output_button
folder_output_button.on_click(on_output_button_click)

# Create an array of button and label widgets
file_selectors = []
for filter_name in filters:
    # Create button and label
    button = widgets.Button(description=f"{filter_name}:")
    if state.get(filter_name):
        label = widgets.Label(value=f" {state.get(filter_name)}")
    else:
        label = widgets.Label(value=f" Not selected")
    
    # Define the button click event
    def on_button_click(filter_name=filter_name, label=label):
        if (filter_name in state.keys()) and (state[filter_name]):
            initialdir=os.path.dirname(state[filter_name][0])
        else:
            initialdir='/'
        selected_dir = open_file_selection(initialdir=initialdir,filter_name=filter_name)  # Adjust initialdir as needed
        if selected_dir:
            label.value = f" {selected_dir[0]}"
            state[filter_name]=selected_dir[0]
            update_oposite_selectors(filter_name,excluding_selectors[0],excluding_selectors[1])
            
        else:
            label.value = f" Not selected"
            state[filter_name]=f" Not selected"
        save_state_pickle(state)

    # Attach the event to the button
    button.on_click(lambda b, f=filter_name, l=label: on_button_click(f, l))
    
    # Add the button and label as a horizontal box
    file_selectors.append(widgets.HBox([button, label]))


datepicker = widgets.DatePicker(
    description='Fecha mail',
    disabled=False,
    value=date.today()-timedelta(days=1)
)

ui = widgets.VBox([datepicker]+[widgets.HBox([folder_output_button, folder_output_label])]+file_selectors)
display(ui)


for file_selector in file_selectors:
    filter_name=file_selector.children[0].description[:-1]
    if filter_name in excluding_selectors:
        file_selector.children[1].value = 'Not selected'
        state[filter_name]=f" Not selected"
        save_state_pickle(state)




## Explorar outlook
- Obtiene los archivos Korrus adjuntos de las fechas no obtenidas anteriormente

In [9]:

# Obtener los archivos adjuntos de los emails

# Crear folder para adjuntos si no existe
path_attachments=os.path.join(folder_output_label.value, "attachments")
if not os.path.exists(path_attachments):
    os.makedirs(path_attachments)

outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
inbox = outlook.GetDefaultFolder(6)  
date_limit = datepicker.value
date_limit=datetime.combine(date_limit, datetime.min.time())
messages = inbox.Items
messages.Sort("[ReceivedTime]", True)  
# df_korrus_list_new=pd.DataFrame(columns=['file_name','received_time','status'])
for message in messages:
    if not hasattr(message, 'ReceivedTime'):
        continue
    received_time = message.ReceivedTime
    received_time = datetime.fromtimestamp(received_time.timestamp())
    if received_time < date_limit:
        break 
    # print(f"Processing email: {message.Subject} (Received: {received_time})")
    if message.Attachments.Count > 0:
        for attachment in message.Attachments:
            # Save the attachment
            attachment_name = attachment.FileName
            attachment_name=f"{os.path.splitext(attachment_name)[0]}_{received_time.strftime('%Y-%m-%d-%H%M%S')}{os.path.splitext(attachment_name)[1]}"
            # if attachment_name in df_korrus_list['file_name'].values:
            #     continue
            if 'KORRUS' not in attachment_name.upper():
                continue
            path_attachment = os.path.join(path_attachments, attachment_name)
            if  not os.path.exists(path_attachment):
                try:
                    attachment.SaveAsFile(path_attachment)
                except Exception as e:
                    print(f"Error saving attachment: {e}")
                # df_korrus_list_new=pd.concat([df_korrus_list_new,pd.DataFrame(data={'file_name':[attachment_name],'received_time':[received_time],'status':['']})],ignore_index=True)
                    

                # print(f"Saved attachment: {attachment_name} to {path_attachment}")


## Consolidar archivos Korrus

In [8]:
#Crear la lista de archivos descargados para manejar su integracion. Se procesa lo que esta en el folder sin importar su origen
path_attachments_done=os.path.join(path_attachments, "Done")
if not os.path.exists(path_attachments_done):
    os.makedirs(path_attachments_done)

#Status maneja el proceso a realizar en los archivos: r=reprocesar, d=done, e=error
path_korrus_list=os.path.join(folder_output_label.value,'Korrus_list.xlsx')
if os.path.exists(path_korrus_list):
    df_korrus_list=pd.read_excel(path_korrus_list)
else:
    df_korrus_list=pd.DataFrame(columns=['file_name','received_time','status'])
close_xl_if_open(path_korrus_list)

files=os.listdir(path_attachments)
df_korrus_list_new=pd.DataFrame(columns=['file_name'],data=files)
df_korrus_list_new['received_time']=df_korrus_list_new['file_name'].str.extract(r'(\d{4}-\d{2}-\d{2}-\d{6})')
df_korrus_list_new=df_korrus_list_new.merge(df_korrus_list,how='outer',on=['file_name','received_time'])
df_korrus_list_new.dropna(subset=['received_time'],inplace=True)
df_korrus_list_new['status'].fillna('',inplace=True)

# Analizar archivos adjuntos
df_korrus_data_new=pd.DataFrame(columns=['origin_file']+mandatory_cols['Korrus'])
for index, row in df_korrus_list_new[df_korrus_list_new['status'].isin(['','r'])].iterrows():
    filepath=os.path.join(path_attachments,row['file_name'])
    if (row['status']=='r') or (not os.path.exists(filepath)):
        filepath=os.path.join(path_attachments_done,row['file_name'])
    if not os.path.exists(filepath):
            df_korrus_list_new.loc[index,'status']='e'
            show_popup_message(f"El archivo {row['file_name']} no existe")
            continue
    if os.path.splitext(os.path.basename(filepath))[1].lower()==".xlsx":
        df = pd.read_excel(filepath)
    elif os.path.splitext(os.path.basename(filepath))[1].lower()==".csv":
        try:
            df = pd.read_csv(filepath, sep='\t', encoding='utf-16')
        except:
            df = pd.read_csv(filepath, sep=',')
    else:
        df_korrus_list_new.loc[index,'status']='e'
        show_popup_message(f"El archivo {row['file_name']} no pudo ser procesado")
        continue
    if check_mandatory_cols(df.columns,selector_name='Korrus',raise_error=False):
        df['origin_file']=row['file_name']
        df_korrus_data_new=pd.concat([df_korrus_data_new,df])
        df_korrus_data_new=df_korrus_data_new[['origin_file']+mandatory_cols['Korrus']]
        df_korrus_list_new.loc[index,'status']='n'
    else:
        df_korrus_list_new.loc[index,'status']='e'

# Consolidar datos en un solo archivo
path_korrus_data=os.path.join(folder_output_label.value,'Korrus_data.xlsx')
if os.path.exists(path_korrus_data):
    df_korrus_data=pd.read_excel(path_korrus_data)
else:
    df_korrus_data=pd.DataFrame(columns=['origin_file']+mandatory_cols['Korrus'])
close_xl_if_open(path_korrus_data)
df_korrus_data=df_korrus_data[~df_korrus_data['origin_file'].isin(df_korrus_data_new['origin_file'])]
df_korrus_data_new=pd.concat([df_korrus_data, df_korrus_data_new])

# Mover archivos a la carpeta de archivos procesados
for index,row in df_korrus_list_new[df_korrus_list_new['status']=='n'].iterrows():
    df_korrus_list_new.loc[index,'status']='y'
    if os.path.exists(f'{path_attachments}/{row["file_name"]}'):
        shutil.move(f'{path_attachments}/{row["file_name"]}', f'{path_attachments_done}/{row["file_name"]}')
# Eliminar archivos ya procesados del folder de llegada
df_remove=df_korrus_list_new[(df_korrus_list_new['status']=='y') & (df_korrus_list_new['file_name'].isin(files))]
for index,row in df_remove.iterrows():
    if os.path.exists(os.path.join(path_attachments,row['file_name'])):
        os.remove(os.path.join(path_attachments,row['file_name']))
files=os.listdir(path_attachments)
# Guardar datos y lista de archivos procesados
save_df(df_korrus_list_new,filepath=path_korrus_list,sheet_name='Korrus List',index=False)
save_df(df_korrus_data_new,filepath=path_korrus_data,sheet_name='Korrus Data',index=False)